# 대퇴골두 FH v3만 교체하기
기존 브라우저 모델 폴더의 A / 번호 / S1 파일을 그대로 복사하고, **FH 두 모델만 새로 변환**합니다.
이전 폴더와 학습 결과는 수정하지 않습니다. 새 폴더 전체를 PC로 내려받아 최신 웹앱에서 선택하세요.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 현재 웹페이지와 같은 소스 버전 사용
import io, zipfile, urllib.request, sys, subprocess
from pathlib import Path
SOURCE = Path('/content/spine_browser_converter')
SOURCE.mkdir(exist_ok=True)
url = 'https://github.com/kimtaejoon01/spine-annotator/archive/refs/heads/codex/integrated-ai-app.zip'
with urllib.request.urlopen(url) as response:
    archive = zipfile.ZipFile(io.BytesIO(response.read()))
    for item in archive.infolist():
        relative = Path(*Path(item.filename).parts[1:])
        if not relative.parts or item.is_dir(): continue
        target = (SOURCE / relative).resolve()
        if not target.is_relative_to(SOURCE.resolve()): raise ValueError('잘못된 압축 경로')
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(item))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SOURCE/'ai_server/browser-export-requirements.txt')])
sys.path.insert(0, str(SOURCE))
# 열린 런타임에서 재실행해도 이전 변환기가 남지 않도록 갱신합니다.
import importlib
importlib.invalidate_caches()
for module_name in ('ai_server.s1_pair', 'ai_server.onnx_export', 'ai_server.runtime', 'ai_server.export_browser'):
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])


In [ ]:
# 두 폴더만 지정합니다. /config.json 이 들어 있는 실제 run 폴더를 선택하세요.
from pathlib import Path
from ai_server.runtime import normalize_path, read_json
EXISTING_BUNDLE = input('기존 브라우저 변환 폴더 (manifest.json 포함): ').strip()
FH_V3_RUN = input('새 FH v3 학습 결과 run 폴더 (coarse_best.pt 포함): ').strip()
existing_bundle = normalize_path(EXISTING_BUNDLE)
fh_run = normalize_path(FH_V3_RUN)
if not (existing_bundle/'manifest.json').is_file():
    raise ValueError('기존 브라우저 변환 결과 폴더를 지정하세요.')
for name in ('config.json','coarse_best.pt','fine_best.pt','selected_threshold.json'):
    if not (fh_run/name).is_file(): raise FileNotFoundError(str(fh_run/name))
if read_json(fh_run/'config.json').get('version') != 'v3':
    raise ValueError('FH v3의 run 폴더를 선택하세요. v2 폴더가 아닌지 확인하세요.')
print('기존 묶음:', existing_bundle)
print('새 골두:', fh_run)


In [ ]:
from datetime import datetime
import uuid
from ai_server.export_browser import replace_femoral_bundle
OUTPUT = existing_bundle.parent / ('fh_v3_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8])
replace_femoral_bundle(existing_bundle=existing_bundle, fh=fh_run, output=OUTPUT)
print('새 모델 폴더:', OUTPUT)


In [ ]:
# 내려받은 ZIP을 풀고, Annotator의 AI 분석 → 모델 폴더 선택에서 폴더 전체를 고릅니다.
import shutil
from google.colab import files
ZIP = shutil.make_archive('/content/spine_browser_models', 'zip', OUTPUT)
files.download(ZIP)


ZIP을 풀고 AI 분석 → 모델 설정 → 내 모델 폴더 선택에서 **새 폴더 전체**를 선택하세요.
모델 표시가 **FH v3**인지 확인하세요. 기존에 브라우저에 저장해 둔 폴더는 v2일 수 있습니다.
FH에는 노트북의 flip TTA + 슬롯 보정, coarse 전체 영상 재탐색, 확률 등고선 원 피팅, 겹친 두 골두 HC 정책이 적용됩니다.
변환 중 원본 PyTorch와 ONNX 출력을 비교하고, 브라우저에서도 모델별 출력값을 확인합니다.
